# Logistic Regresssion

In [1]:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('https://raw.githubusercontent.com/marvv0905/ISOM3360-project/refs/heads/main/datasets/shopping_dataset_dummyCoded.csv')

### Split df into features and target variable

In [2]:
#encode Y, since multiclass value
encoder = LabelEncoder()

X = df.drop("shopping_preference", axis=1)
Y = encoder.fit_transform(df["shopping_preference"])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=42)


### Normalizing the numerical features

In [3]:
#exclude the binary features
dummy_cols = ['gender_Male', 'gender_Female', 'gender_Other','city_tier_Tier 1', 'city_tier_Tier 2', 'city_tier_Tier 3']
numeric_cols = [col for col in X.columns if col not in dummy_cols]

normalize = preprocessing.StandardScaler().fit(X_train[numeric_cols])
X_train[numeric_cols] = normalize.transform(X_train[numeric_cols])
X_test[numeric_cols] = normalize.transform(X_test[numeric_cols])

## Base Model

In [4]:
logistic_model_base = LogisticRegression().fit(X_train, y_train)

In [9]:
test_performance  = logistic_model_base.predict(X_test)
print("Test set performance")
print(classification_report(y_test, test_performance, target_names=encoder.classes_))
print(confusion_matrix(y_test, test_performance))

train_score = logistic_model_base.score(X_train, y_train)
test_score = logistic_model_base.score(X_test, y_test)

print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy:  {test_score:.4f}")

Test set performance
              precision    recall  f1-score   support

      Hybrid       0.89      0.36      0.51       111
      Online       0.93      0.99      0.96       353
       Store       0.99      1.00      0.99      3073

    accuracy                           0.98      3537
   macro avg       0.94      0.78      0.82      3537
weighted avg       0.98      0.98      0.97      3537

[[  40   25   46]
 [   5  348    0]
 [   0    0 3073]]
Train accuracy: 0.9827
Test accuracy:  0.9785


## Model Tuning 

### 1. Lasso regularlization

In [10]:
logistic_model_l1 = LogisticRegression(penalty='l1', C=1, max_iter=10000,
                                    random_state=0,
                                    solver='saga', 
                                    class_weight='balanced', 
                                    tol=0.01)

logistic_model_l1.fit(X_train, y_train)
logistic_l1_train_perform = logistic_model_l1.predict(X_train)
logistic_l1_test_perform = logistic_model_l1.predict(X_test)

print("Testing set performance")
print(classification_report(y_test, logistic_l1_test_perform, target_names=encoder.classes_))

print(f"Train accuracy: {logistic_model_l1.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {logistic_model_l1.score(X_test, y_test):.4f}")

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.52      0.77      0.62       111
      Online       0.97      0.95      0.96       353
       Store       1.00      0.98      0.99      3073

    accuracy                           0.97      3537
   macro avg       0.83      0.90      0.86      3537
weighted avg       0.98      0.97      0.97      3537

Train accuracy: 0.9719
Test accuracy:  0.9703


#### Finding best 'c' for L1 model with GridsearchCV

In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_logisticReg_l1 = GridSearchCV(estimator=LogisticRegression(penalty='l1', solver='saga',
                                  max_iter=40000, tol=0.01,
                                  class_weight='balanced',
                                  random_state=0),
                                  param_grid=param_grid,
                                  cv=10,
                                  scoring='f1_macro'
)

grid_logisticReg_l1.fit(X_train, y_train)

print("Best C:", grid_logisticReg_l1.best_params_)
print("Best CV Macro F1:", grid_logisticReg_l1.best_score_)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max

Best C: {'C': 100}
Best CV Macro F1: 0.8760644822931356


#### Applying the best 'c' for L1 model

In [14]:
logistic_model_l1 = LogisticRegression(penalty='l1', C=100, max_iter=10000,
                                    random_state=0,
                                    solver='saga', 
                                    class_weight='balanced', 
                                    tol=0.01)

logistic_model_l1.fit(X_train, y_train)
logistic_l1_train_perform = logistic_model_l1.predict(X_train)
logistic_l1_test_perform = logistic_model_l1.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l1_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l1_test_perform, target_names=encoder.classes_))

print(f"Train accuracy: {logistic_model_l1.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {logistic_model_l1.score(X_test, y_test):.4f}")


Training set performance
              precision    recall  f1-score   support

      Hybrid       0.52      0.94      0.67       258
      Online       0.98      0.94      0.96       823
       Store       1.00      0.98      0.99      7171

    accuracy                           0.97      8252
   macro avg       0.83      0.95      0.87      8252
weighted avg       0.98      0.97      0.98      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.55      0.95      0.70       111
      Online       0.99      0.95      0.97       353
       Store       1.00      0.98      0.99      3073

    accuracy                           0.97      3537
   macro avg       0.85      0.96      0.89      3537
weighted avg       0.98      0.97      0.98      3537

Train accuracy: 0.9710
Test accuracy:  0.9743


### 2. Ridge regularlization

In [9]:
logistic_model_l2 = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                                    max_iter=1000,
                                    class_weight='balanced',
                                    random_state=0)


logistic_model_l2.fit(X_train, y_train)
logistic_l2_train_perform = logistic_model_l2.predict(X_train)
logistic_l2_test_perform = logistic_model_l2.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l2_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l2_test_perform, target_names=encoder.classes_))

print("===========================================================")

logistic_score_l2 = cross_val_score(logistic_model_l2, X_train, y_train, cv=10, scoring='f1_macro')
print(f"CV Macro F1: {logistic_score_l2.mean():.4f} ± {logistic_score_l2.std():.4f}")

Training set performance
              precision    recall  f1-score   support

      Hybrid       0.40      1.00      0.57       258
      Online       1.00      0.94      0.97       823
       Store       1.00      0.95      0.98      7171

    accuracy                           0.95      8252
   macro avg       0.80      0.96      0.84      8252
weighted avg       0.98      0.95      0.96      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.43      1.00      0.60       111
      Online       1.00      0.94      0.97       353
       Store       1.00      0.96      0.98      3073

    accuracy                           0.96      3537
   macro avg       0.81      0.97      0.85      3537
weighted avg       0.98      0.96      0.97      3537

CV Macro F1: 0.8315 ± 0.0156


#### Finding best 'c' for L2 model with GridsearchCV

In [10]:
grid_logisticReg_l2 = GridSearchCV(estimator=LogisticRegression(penalty='l2', solver='lbfgs',
                                  max_iter=1000,
                                  class_weight='balanced',
                                  random_state=0),
                                  param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100]},
                                  cv=5,
                                  scoring='f1_macro'
)

grid_logisticReg_l2.fit(X_train, y_train)


print("Best C:", grid_logisticReg_l2.best_params_)
print("Best CV Macro F1:", grid_logisticReg_l2.best_score_)

Best C: {'C': 100}
Best CV Macro F1: 0.9351099650234538


In [12]:
logistic_model_l2 = LogisticRegression(penalty='l2', C=100, solver='lbfgs',
                                    max_iter=1000,
                                    class_weight='balanced',
                                    random_state=0)

logistic_model_l2.fit(X_train, y_train)
logistic_l2_train_perform = logistic_model_l2.predict(X_train)
logistic_l2_test_perform = logistic_model_l2.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l2_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l2_test_perform, target_names=encoder.classes_))

print("===========================================================")

print(f"Train accuracy: {logistic_model_l2.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {logistic_model_l2.score(X_test, y_test):.4f}")

Training set performance
              precision    recall  f1-score   support

      Hybrid       0.74      1.00      0.85       258
      Online       1.00      0.99      0.99       823
       Store       1.00      0.99      0.99      7171

    accuracy                           0.99      8252
   macro avg       0.91      0.99      0.95      8252
weighted avg       0.99      0.99      0.99      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.71      0.99      0.83       111
      Online       1.00      0.97      0.99       353
       Store       1.00      0.99      0.99      3073

    accuracy                           0.99      3537
   macro avg       0.90      0.98      0.94      3537
weighted avg       0.99      0.99      0.99      3537

Train accuracy: 0.9891
Test accuracy:  0.9873
